In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
storage_account = "pharamadatalake"
silver_base_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/"
gold_base_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/"

In [0]:
#Dimention Tables

df_ct_silver = spark.read.format("delta").load(f"{silver_base_path}clinical_trials_silver/")

# Check available columns
print("Available columns in clinical_trials_silver:")
print(df_ct_silver.columns)

dim_clinical_trials = df_ct_silver.withColumn("trial_sk", monotonically_increasing_id())\
    .withColumn("effective_date", current_date())\
    .withColumn("is_current", lit(True))\
        .select("trial_sk","trial_id","trial_phase_standardized", "trial_status_standardized","sponsor","patient_count","enrollment_category","is_completed","is_ongoing","trial_year","trial_month","effective_date","is_current")

dim_clinical_trials.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base_path}dim_clinical_trials/")
print(f"  ✓ Dim_Clinical_Trials: {dim_clinical_trials.count()} records")


Available columns in clinical_trials_silver:
['trial_id', 'drug_name', 'condition_treated', 'trial_phase', 'start_date', 'end_date', 'patient_count', 'male_participants', 'female_participants', 'age_range', 'efficacy_rate', 'placebo_efficacy', 'adverse_events_count', 'serious_adverse_events', 'trial_status', 'sponsor', 'budget_millions', 'completion_rate', 'data_quality_score', 'publication_status', 'ingestion_timestamp', 'source_file', 'ingestion_batch_id', 'trial_phase_standardized', 'trial_status_standardized', 'is_completed', 'is_ongoing', 'trial_year', 'trial_month', 'enrollment_category', 'silver_processing_timestamp', 'silver_batch_id']
  ✓ Dim_Clinical_Trials: 500 records


In [0]:
df_drugs = spark.read.format("delta").load(f"{silver_base_path}sales_silver/")\
 .select("drug_name")\
 .distinct()\
 .withColumn("drug_sk", monotonically_increasing_id())\
 .withColumn("effective_date", current_date())\
 .withColumn("is_current", lit(True))

dim_drugs = df_drugs.select(
 "drug_sk",
"drug_name",
 "effective_date",
 "is_current"
)

dim_drugs.write.format("delta") \
 .mode("overwrite") \
 .save(f"{gold_base_path}dim_drugs/")
print(f" ✓ Dim_Drugs: {dim_drugs.count()} records")


 ✓ Dim_Drugs: 100 records


In [0]:
df_physician_silver = spark.read.format("delta").load(f"{silver_base_path}physician_silver/")

# Check available columns
print("Available columns in physician_silver:")
print(df_physician_silver.columns)

dim_physicians = df_physician_silver\
    .withColumn("physician_sk", monotonically_increasing_id())\
    .withColumn("effective_date", current_date())\
    .withColumn("is_current", lit(True))\
    .select(
     "physician_sk",
     "physician_id",
     "physician_name",
     "specialty",
     "years_experience",
     "experience_level",
     "hospital_affiliation",
     col("state") if "state" in df_physician_silver.columns else lit("Unknown").alias("state"),
     "prescription_volume_monthly",
     "most_prescribed_category",
     "patient_satisfaction_score",
     "is_current"
 )

dim_physicians.write.format("delta") \
     .mode("overwrite") \
    .save(f"{gold_base_path}dim_physicians/")
print(f"  ✓ Dim_Physicians: {dim_physicians.count()} records")


Available columns in physician_silver:
['physician_id', 'physician_name', 'specialty', 'years_experience', 'hospital_affiliation', 'practice_type', 'state', 'prescription_volume_monthly', 'most_prescribed_category', 'preferred_brands', 'e-prescribing_usage', 'pharma_visits_monthly', 'sample_drugs_accepted', 'off_label_prescribing_rate', 'patient_satisfaction_score', 'alcohol_prescription_rate', 'ingestion_timestamp', 'source_file', 'ingestion_batch_id', 'experience_level', 'e_prescribing_flag', 'sample_drugs_flag', 'prescription_volume_category', 'satisfaction_category', 'silver_processing_timestamp', 'silver_batch_id']
  ✓ Dim_Physicians: 700 records


In [0]:
def create_date_dimension(start_date, end_date):
    dates = spark.sql(f"""
     SELECT EXPLODE(SEQUENCE(
     TO_DATE('{start_date}'),
     TO_DATE('{end_date}'),
    INTERVAL 1 DAY
     )) AS date
     """)

    date_dim = dates.select(
     col("date"),
    year("date").alias("year"),
     quarter("date").alias("quarter"),
    month("date").alias("month"),
    dayofmonth("date").alias("day"),
     dayofweek("date").alias("day_of_week"),
    date_format("date", "EEEE").alias("day_name"),
     date_format("date", "MMMM").alias("month_name"),
     when(dayofweek("date").isin([1, 7]), "Weekend").otherwise("Weekday").alias("weekday_indicator")
    ).withColumn("date_sk", monotonically_increasing_id())
 
    return date_dim

dim_date = create_date_dimension("2020-01-01", "2026-12-31")

dim_date.write.format("delta") \
     .mode("overwrite") \
     .save(f"{gold_base_path}dim_date/")
print(f"  ✓ Dim_Date: {dim_date.count()} records")


  ✓ Dim_Date: 2557 records


In [0]:
# Fact Tables 


df_sales_silver = spark.read.format("delta").load(f"{silver_base_path}sales_silver/")
print("Available columns in sales_silver:")
print(df_sales_silver.columns)

dim_drugs = spark.read.format("delta").load(f"{gold_base_path}dim_drugs/")
dim_date = spark.read.format("delta").load(f"{gold_base_path}dim_date/")


Available columns in sales_silver:
['sale_id', 'drug_name', 'region', 'country', 'sales_channel', 'sales_date', 'units_sold', 'unit_price_usd', 'total_revenue_usd', 'discount_percent', 'net_revenue_usd', 'sales_rep_id', 'customer_type', 'insurance_coverage', 'promotion_applied', 'season', 'ingestion_timestamp', 'source_file', 'ingestion_batch_id', 'sales_channel_standardized', 'sales_year', 'sales_month', 'sales_quarter', 'is_high_value', 'revenue_category', 'silver_processing_timestamp', 'silver_batch_id']


In [0]:
fact_sales = df_sales_silver \
    .join(dim_drugs.select("drug_name", "drug_sk"), "drug_name", "left") \
     .join(dim_date.select("date", "date_sk"), df_sales_silver["sales_date"] == dim_date["date"], "left") \
     .select(
     "sale_id",
    "drug_sk",
    "date_sk",
     "sales_channel_standardized",
    "units_sold",
     "unit_price_usd",
     "total_revenue_usd",
    "region",
     "country" if "country" in df_sales_silver.columns else lit("Unknown").alias("country"),
     "sales_rep_id" if "sales_rep_id" in df_sales_silver.columns else lit("Unknown").alias("sales_rep_id"),
     "sales_year",
     "sales_month",
     "sales_quarter",
     "is_high_value",
     "revenue_category"
)

fact_sales.write.format("delta") \
    .mode("overwrite") \
     .option("overwriteSchema", "true") \
    .save(f"{gold_base_path}fact_sales/")
print(f"  ✓ Fact_Sales: {fact_sales.count()} records")


  ✓ Fact_Sales: 1000 records


In [0]:
f_ct_silver = spark.read.format("delta").load(f"{silver_base_path}clinical_trials_silver/")
dim_clinical_trials = spark.read.format("delta").load(f"{gold_base_path}dim_clinical_trials/")
dim_date = spark.read.format("delta").load(f"{gold_base_path}dim_date/")

# Check available columns
print("Available columns in clinical_trials_silver for fact:")
print(df_ct_silver.columns)

fact_clinical_trials = df_ct_silver \
     .join(dim_clinical_trials.select("trial_id", "trial_sk"), "trial_id", "left") \
     .join(dim_date.select("date", "date_sk"), df_ct_silver["start_date"] == dim_date["date"], "left") \
     .select(
     "trial_id",
    "trial_sk",
     "date_sk",
     "drug_name",
     "condition_treated",
     "patient_count",
     "efficacy_rate",
     "placebo_efficacy" if "placebo_efficacy" in df_ct_silver.columns else lit(0.0).alias("placebo_efficacy"),
     "adverse_events_count",
     "serious_adverse_events" if "serious_adverse_events" in df_ct_silver.columns else lit(0).alias("serious_adverse_events"),
     "budget_millions" if "budget_millions" in df_ct_silver.columns else lit(0.0).alias("budget_millions"),
     "completion_rate" if "completion_rate" in df_ct_silver.columns else lit(0.0).alias("completion_rate"),
     "data_quality_score" if "data_quality_score" in df_ct_silver.columns else lit(0.0).alias("data_quality_score")
     )
fact_clinical_trials.write.format("delta") \
    .mode("overwrite") \
     .option("overwriteSchema", "true") \
    .save(f"{gold_base_path}fact_clinical_trials/")
print(f"  ✓ Fact_Clinical_Trials: {fact_clinical_trials.count()} records")


Available columns in clinical_trials_silver for fact:
['trial_id', 'drug_name', 'condition_treated', 'trial_phase', 'start_date', 'end_date', 'patient_count', 'male_participants', 'female_participants', 'age_range', 'efficacy_rate', 'placebo_efficacy', 'adverse_events_count', 'serious_adverse_events', 'trial_status', 'sponsor', 'budget_millions', 'completion_rate', 'data_quality_score', 'publication_status', 'ingestion_timestamp', 'source_file', 'ingestion_batch_id', 'trial_phase_standardized', 'trial_status_standardized', 'is_completed', 'is_ongoing', 'trial_year', 'trial_month', 'enrollment_category', 'silver_processing_timestamp', 'silver_batch_id']
  ✓ Fact_Clinical_Trials: 500 records
